In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from experiments import Experiment, fitting_time, svd_time, normalization_time, number_of_features
from journal2026 import (
    JOURNAL2026_TRAIN_SIZES,
    JOURNAL2026_D1_REGULAR, JOURNAL2026_D1_LARGE,
    TIMING_ESTIMATORS, TIMING_EST_NAMES,
)

In [ ]:
exp_regular = Experiment(
    JOURNAL2026_D1_REGULAR, TIMING_ESTIMATORS,
    reps=30, ns=[[JOURNAL2026_TRAIN_SIZES[p.dataset]] for p in JOURNAL2026_D1_REGULAR],
    seed=123,
    stats=[fitting_time, svd_time, normalization_time, number_of_features],
    est_names=TIMING_EST_NAMES).run()

In [ ]:
exp_large = Experiment(
    JOURNAL2026_D1_LARGE, TIMING_ESTIMATORS,
    reps=30, ns=[[JOURNAL2026_TRAIN_SIZES[p.dataset]] for p in JOURNAL2026_D1_LARGE],
    seed=123,
    stats=[fitting_time, svd_time, normalization_time, number_of_features],
    est_names=TIMING_EST_NAMES).run()

In [ ]:
for exp, label in [(exp_regular, 'regular'), (exp_large, 'large')]:
    datasets = [p.dataset for p in exp.problems]
    for i, ds in enumerate(datasets):
        norm_vals = np.stack([exp.normalization_time_[:, i, 0, j]
                              for j in range(len(TIMING_EST_NAMES))])
        svd_vals  = np.stack([exp.svd_time_[:, i, 0, j]
                              for j in range(len(TIMING_EST_NAMES))])
        for name, vals in [('norm', norm_vals), ('svd', svd_vals)]:
            means = vals.mean(axis=1)
            overall_mean = means.mean()
            if overall_mean > 0 and means.std() / overall_mean > 0.10:
                print(f'WARNING [{label}] {ds}: {name} std/mean={means.std()/overall_mean:.2f}')
print('Sanity check complete')

In [ ]:
def timing_table(exp):
    rows = []
    for i, prob in enumerate(exp.problems):
        t_em    = exp.fitting_time_[:, i, 0, 0].mean()
        t_cv101 = exp.fitting_time_[:, i, 0, 1].mean()
        t_cv11  = exp.fitting_time_[:, i, 0, 2].mean()
        t_svd   = np.stack([exp.svd_time_[:, i, 0, j]
                            for j in range(3)]).mean()
        t_norm  = np.stack([exp.normalization_time_[:, i, 0, j]
                            for j in range(3)]).mean()
        t_prep  = t_svd + t_norm
        rows.append({
            'dataset': prob.dataset,
            'T_EM':    round(t_em,    4),
            'T_CV101': round(t_cv101, 4),
            'T_CV11':  round(t_cv11,  4),
            'T_svd':   round(t_svd,   4),
            'T_norm':  round(t_norm,  4),
            'T_prep':  round(t_prep,  4),
            'SU':      round(t_cv101 / t_em, 2),
            'SU_post': round((t_cv101 - t_prep) / (t_em - t_prep), 2)
                       if (t_em - t_prep) > 0 else float('nan'),
        })
    return pd.DataFrame(rows).set_index('dataset')

In [ ]:
timing_table(exp_regular)

In [ ]:
timing_table(exp_large)

In [ ]:
colors_norm = ['#d9534f', '#e87e77', '#f5b8b5']
colors_svd  = ['#5bc0de', '#7ccfe8', '#b3e4f5']
colors_fit  = ['#5cb85c', '#7fca7f', '#b3e4b3']
width = 0.25
offsets = [-width, 0, width]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, exp, title in [(axes[0], exp_regular, 'Regular datasets'),
                        (axes[1], exp_large,   'Large datasets')]:
    datasets = [p.dataset for p in exp.problems]
    n = len(datasets)
    x = np.arange(n)
    for j, name in enumerate(TIMING_EST_NAMES):
        t_norm_mean = exp.normalization_time_[:, :, 0, j].mean(axis=0)
        t_svd_mean  = exp.svd_time_[:, :, 0, j].mean(axis=0)
        t_fit_mean  = (exp.fitting_time_[:, :, 0, j].mean(axis=0)
                       - t_svd_mean - t_norm_mean)
        t_total     = exp.fitting_time_[:, :, 0, j]
        ci_lo = np.percentile(t_total, 2.5, axis=0)
        ci_hi = np.percentile(t_total, 97.5, axis=0)
        t_mean = t_total.mean(axis=0)
        xpos = x + offsets[j]
        ax.bar(xpos, t_norm_mean, width=width, color=colors_norm[j], label=f'{name}: T_norm')
        ax.bar(xpos, t_svd_mean,  width=width, color=colors_svd[j],
               bottom=t_norm_mean, label=f'{name}: T_svd')
        ax.bar(xpos, t_fit_mean,  width=width, color=colors_fit[j],
               bottom=t_norm_mean + t_svd_mean, label=f'{name}: T_fit')
        ax.errorbar(xpos, t_mean,
                    yerr=[t_mean - ci_lo, ci_hi - t_mean],
                    fmt='none', color='black', capsize=3, linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha='right')
    ax.set_ylabel('Time [s]')
    ax.set_title(title)

axes[0].legend(loc='upper right', fontsize=7)
plt.tight_layout()